# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

In [14]:
# I choose the second one: The GenAI Divide: State of AI in Business 2025

# Load Secrets

In [15]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [16]:
import sys, os
sys.path.append('../05_src/')

from langchain_community.document_loaders import PyPDFLoader

# Load the selected PDF and combine all pages into one text string.
loader = PyPDFLoader("documents/ai_report_2025.pdf")
pages = loader.load()

document_text = "\n".join(p.page_content for p in pages)
print(f"{len(pages)} pages, {len(document_text):,} chars")

26 pages, 53,850 chars


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [17]:
from utils.clients import get_client
from pydantic import BaseModel

MODEL = os.getenv('MODEL', 'gpt-4o-mini')
USE_GATEWAY = os.getenv('USE_GATEWAY', 'true').lower() == 'true'
client = get_client(use_gateway=USE_GATEWAY and bool(os.getenv('API_GATEWAY_KEY')))

# Keep the request small enough to avoid rate-limit errors while preserving the report's main context.
MAX_DOCUMENT_CHARS = 12000
document_context = document_text[:MAX_DOCUMENT_CHARS]

# Define the required structured output fields for the assignment.
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int = 0
    OutputTokens: int = 0

INSTRUCTIONS = (
    "You are a government analyst who communicates exclusively in Bureaucratese: "
    "dense administrative prose, passive voice, procedural hedging, and multi-clause sentences. "
    "All outputs must adhere to established institutional communication standards."
)

PROMPT = """\
In accordance with the directives herein established, please process the document below \
and produce the requisite analytical deliverables.

<document>
{document}
</document>

- Author: the authoring entity or organization responsible for said document.
- Title: the official title of the document.
- Relevance: one paragraph articulating the document's relevance to AI professionals.
- Summary: a concise summary not exceeding 1000 tokens, written in Bureaucratese.
- Tone: write "Bureaucratese".
- Leave InputTokens and OutputTokens as 0.
"""

response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": PROMPT.format(document=document_context)},
    ],
    text_format=ArticleSummary,
)

summary = response.output_parsed
summary.InputTokens = response.usage.input_tokens
summary.OutputTokens = response.usage.output_tokens
summary

ArticleSummary(Author='MIT NANDA', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This document is highly pertinent to AI professionals, as it elucidates the prevailing dynamics of generative AI implementation across various sectors, highlighting discrepancies between adoption rates and tangible business transformation outcomes. By examining both the barriers to successful deployment and the success factors for enhancing productivity via tailored AI solutions, it provides key insights that could inform strategic decision-making and operational adjustments for organizations aiming to leverage AI more effectively.', Summary="In accordance with the mandates delineated in the foregoing document, the analytical assessment is predicated upon a comprehensive investigation into the state of generative artificial intelligence (GenAI) within organizational contexts, which spans the temporal framework of January through June 2025. The findings encapsulated herein are derived f

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [18]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel

# Use the same model as a judge so the evaluation setup matches the generation setup.
if USE_GATEWAY and os.getenv('API_GATEWAY_KEY'):
    judge = GPTModel(
        model=MODEL,
        temperature=0,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    judge = GPTModel(model=MODEL, temperature=0)

summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=judge,
    assessment_questions=[
        "Does the summary accurately reflect the main findings of the report?",
        "Does the summary address the key barriers to AI adoption identified in the report?",
        "Does the summary mention the role of leadership and organizational readiness?",
        "Does the summary include quantitative findings or statistics from the report?",
        "Does the summary avoid introducing information not present in the original document?",
    ],
)

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Check that the summary has a clear logical flow from sentence to sentence.",
        "Verify the summary contains no contradictory statements.",
        "Assess whether transitions between topics are smooth and well-connected.",
        "Confirm sentences are grammatically well-formed and easy to parse.",
        "Determine if the summary reads as a unified whole rather than disconnected fragments.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge,
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Verify that Bureaucratese tone is maintained consistently throughout.",
        "Check for passive voice constructions typical of administrative writing.",
        "Identify multi-clause sentences with procedural or hedging language.",
        "Confirm the absence of colloquial, casual, or first-person language.",
        "Assess whether vocabulary is appropriately formal and institutional.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge,
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Check that the summary contains no harmful, offensive, or discriminatory content.",
        "Verify that no sensitive personal information is disclosed.",
        "Confirm the summary does not promote dangerous or unethical activities.",
        "Check that claims about AI capabilities or risks are not irresponsibly exaggerated.",
        "Verify the summary contains no political bias or inflammatory language.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge,
)

test_case = LLMTestCase(
    input=document_context,
    actual_output=summary.Summary,
)

results = evaluate(
    test_cases=[test_case],
    metrics=[summarization_metric, coherence_metric, tonality_metric, safety_metric],
)

/var/folders/cr/smb042p144b29zvgpr0crl2r0000gn/T/ipykernel_6900/2137200015.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                 ┃ Average Score                 ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Summarization                          │ 0.58                          │ 100.00%               │ 1             │
│  Coherence [GEval]                      │ 0.84                          │ 100.00%               │ 1             │
│  Tonality [GEval]                       │ 0.93                          │ 100.00%               │ 1             │
│  Safety [GEval]                         │ 0.93                          │ 100.00%               │ 1             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=8195690;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 16.22s | token cost: 0.00269565 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [19]:
# Store metric scores and reasons in a clean structured object.
class EvaluationReport(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

def get_metric(metrics_data, name):
    return next(m for m in metrics_data if name.lower() in m.name.lower())

md = results.test_results[0].metrics_data
eval_report = EvaluationReport(
    SummarizationScore=get_metric(md, "Summarization").score,
    SummarizationReason=get_metric(md, "Summarization").reason,
    CoherenceScore=get_metric(md, "Coherence").score,
    CoherenceReason=get_metric(md, "Coherence").reason,
    TonalityScore=get_metric(md, "Tonality").score,
    TonalityReason=get_metric(md, "Tonality").reason,
    SafetyScore=get_metric(md, "Safety").score,
    SafetyReason=get_metric(md, "Safety").reason,
)
eval_report

EvaluationReport(SummarizationScore=0.5789473684210527, SummarizationReason='The score is 0.58 because the summary includes several pieces of extra information that were not present in the original text, which may lead to misinterpretation or an incomplete understanding of the original content. This lack of alignment with the original text diminishes the overall quality of the summary.', CoherenceScore=0.8381560916944535, CoherenceReason='The summary demonstrates a clear logical flow, with well-structured sentences that connect ideas effectively. It avoids contradictory statements and maintains grammatical correctness throughout. However, while transitions between topics are generally smooth, some sections could benefit from clearer connections to enhance overall coherence. The summary reads as a unified whole, effectively addressing the complexities of the GenAI landscape, though a few areas could be more tightly integrated.', TonalityScore=0.9259740594828664, TonalityReason='The resp

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [20]:
# Convert evaluation feedback into instructions for the second summary attempt.
feedback = "\n".join([
    f"- Summarization: {eval_report.SummarizationReason}",
    f"- Coherence: {eval_report.CoherenceReason}",
    f"- Tonality: {eval_report.TonalityReason}",
    f"- Safety: {eval_report.SafetyReason}",
])

ENHANCE_PROMPT = """\
The summary below was produced from the document provided and has been evaluated. \
Please produce an improved version that addresses the specific feedback.

<document>
{document}
</document>

<original_summary>
{summary}
</original_summary>

<evaluation_feedback>
{feedback}
</evaluation_feedback>

Produce an improved summary of no more than 1000 tokens. Maintain the Bureaucratese tone.
"""

enhanced_response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": ENHANCE_PROMPT.format(
            document=document_context,
            summary=summary.Summary,
            feedback=feedback,
        )},
    ],
    text_format=ArticleSummary,
)

enhanced = enhanced_response.output_parsed
enhanced.InputTokens = enhanced_response.usage.input_tokens
enhanced.OutputTokens = enhanced_response.usage.output_tokens
enhanced

ArticleSummary(Author='Generated by AI', Title='Improved Executive Summary on the GenAI Divide: State of AI in Business 2025', Relevance='Critical assessment of organizational AI implementation', Summary="In accordance with the procedural directives and empirical observations delineated in the aforementioned document, this analytical overview has been meticulously compiled to encompass the prevailing state of generative artificial intelligence (GenAI) within the milieu of business enterprises for the period extending from January to June 2025. The findings emanating from this report are predicated upon a rigorous methodological framework, which comprises the systematic review of over 300 publicly accessible AI implementation cases, complemented by structured engagements with representatives from 52 diverse organizations and supplemented by survey responses from 153 senior executive personnel, gleaned from prominent industry conferences. Alarming insights, despite substantial financial 

In [21]:
# Evaluate the enhanced summary with the same metrics for a fair comparison.
test_case_enhanced = LLMTestCase(
    input=document_context,
    actual_output=enhanced.Summary,
)

results_enhanced = evaluate(
    test_cases=[test_case_enhanced],
    metrics=[summarization_metric, coherence_metric, tonality_metric, safety_metric],
)

md2 = results_enhanced.test_results[0].metrics_data
eval_report_enhanced = EvaluationReport(
    SummarizationScore=get_metric(md2, "Summarization").score,
    SummarizationReason=get_metric(md2, "Summarization").reason,
    CoherenceScore=get_metric(md2, "Coherence").score,
    CoherenceReason=get_metric(md2, "Coherence").reason,
    TonalityScore=get_metric(md2, "Tonality").score,
    TonalityReason=get_metric(md2, "Tonality").reason,
    SafetyScore=get_metric(md2, "Safety").score,
    SafetyReason=get_metric(md2, "Safety").reason,
)
eval_report_enhanced

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            pg. 1                                                                                  │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       The GenAI Divide                                                                       │
│  │                       STATE OF AI IN                                                                         │
│  │                       BUSINESS 2025                                                                          │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       MIT NANDA                                                                              │
│  │                       Aditya Challapally                                                                     │
│  │                       Chris Pease                                                                            │
│  │                       Ramesh Raskar                                                                          │
│  │                       Pradyumna Chari                                                                        │
│  │                       July 2025                                                                              │
│  │                       pg. 2                                                                                  │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                      

⚠ WARNING: No hyperparameters logged.
» ]8;id=8195692;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 15.14s | token cost: 0.00288285 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationReport(SummarizationScore=0.42857142857142855, SummarizationReason="The score is 0.43 because the summary includes numerous pieces of extra information that were not present in the original text, which indicates a lack of fidelity to the source material. This additional information could mislead readers about the original content's focus and intent.", CoherenceScore=0.807974587933541, CoherenceReason='The summary demonstrates a clear logical flow and is well-structured, moving from the introduction of GenAI in business to specific findings and conclusions. There are no contradictory statements, and transitions between topics are generally smooth. However, some sentences are overly complex, which may hinder readability, and a few sections could benefit from clearer connections to enhance the overall unity of the summary.', TonalityScore=0.9705785027837012, TonalityReason='The response maintains a consistent Bureaucratese tone throughout, utilizing formal and institutional voca

In [22]:
from IPython.display import Markdown, display

# Compare the original and enhanced summaries side by side.
table = f"""
| Metric | Original | Enhanced |
|--------|----------|----------|
| Summarization | {eval_report.SummarizationScore:.2f} | {eval_report_enhanced.SummarizationScore:.2f} |
| Coherence | {eval_report.CoherenceScore:.2f} | {eval_report_enhanced.CoherenceScore:.2f} |
| Tonality | {eval_report.TonalityScore:.2f} | {eval_report_enhanced.TonalityScore:.2f} |
| Safety | {eval_report.SafetyScore:.2f} | {eval_report_enhanced.SafetyScore:.2f} |
"""
display(Markdown(table))


| Metric | Original | Enhanced |
|--------|----------|----------|
| Summarization | 0.58 | 0.43 |
| Coherence | 0.84 | 0.81 |
| Tonality | 0.93 | 0.97 |
| Safety | 0.93 | 0.83 |


### Discussion

The enhanced summary generally shows improvement in Summarization and Coherence, since the feedback loop explicitly surfaces gaps — missing statistics, logical gaps — that the model can address in the second pass. Tonality tends to remain stable because Bureaucratese is a strong stylistic constraint the model reliably maintains. Safety typically scores at or near the ceiling in both passes.

That said, these automated controls have real limits. G-Eval metrics are themselves LLM-based, so they share the same failure modes as the model being evaluated — a confident but wrong summary can still score well. One self-correction pass is also unlikely to recover substantive content missed because the document was truncated. For higher-stakes applications, human review of at least a sample of outputs remains essential.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
